# Lab 11A — Unity Catalog: Gestion de Objetos y Control de Acceso

**Sesion 11 | Databricks Data Engineer Associate**  

**Runtime minimo:** DBR 13.3 LTS  
**Archivo fuente:** `empleados_empresa.csv`

## Objetivos
- Explorar la jerarquia de Unity Catalog con comandos SHOW y DESCRIBE
- Crear tablas gestionadas (managed) y externas (external) en UC
- Otorgar y revocar permisos con GRANT y REVOKE
- Auditar permisos con SHOW GRANTS e information_schema

## Setup previo
Subir `empleados_empresa.csv` al Volume:  
`/Volumes/dbassociate/default/vol_landing/sesion11/`

## Paso 0 — Verificacion del entorno

In [0]:
# Verificar que el archivo fuente existe en el volume
display(dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/"))

In [0]:
CATALOG       = "dominio_data"
SCHEMA_BRONZE = "bronze"
VOLUME_PATH   = "/Volumes/dbassociate/default/vol_landing/sesion11"
SOURCE_FILE   = f"{VOLUME_PATH}/empleados_empresa.csv"
TABLE_PATH = "abfss://dominio-data@stacadbwus01.dfs.core.windows.net/external_tables"

print(f"Catalog  : {CATALOG}")
print(f"Schema   : {SCHEMA_BRONZE}")
print(f"Archivo  : {SOURCE_FILE}")
print(f"External Loc  : {TABLE_PATH}")

## Paso 1 — Explorar la jerarquia de Unity Catalog

In [0]:
# 1A: Listar todos los catalogs del metastore
spark.sql("SHOW CATALOGS").display()

In [0]:
# 1B: Listar schemas dentro del catalog dbassociate
spark.sql("SHOW SCHEMAS IN dominio_data").display()

In [0]:
# 1C: Listar tablas en el schema bronze
spark.sql("SHOW TABLES IN dbassociate.bronze").display()

In [0]:
# 1D: Describir el catalog — ver propietario y comentario
spark.sql("DESCRIBE CATALOG dbassociate").display()

In [0]:
# 1E: Describir el schema con detalle extendido
spark.sql("DESCRIBE SCHEMA EXTENDED dbassociate.bronze").display()

## Paso 2 — Cargar datos de empleados

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema_empleados = StructType([
    StructField("empleado_id",   StringType(), False),
    StructField("nombre",        StringType(), True),
    StructField("apellido",      StringType(), True),
    StructField("email",         StringType(), True),
    StructField("telefono",      StringType(), True),
    StructField("departamento",  StringType(), True),
    StructField("cargo",         StringType(), True),
    StructField("salario",       DoubleType(), True),
    StructField("fecha_ingreso", StringType(), True),
    StructField("pais",          StringType(), True),
])

df_empleados = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .schema(schema_empleados)
    .csv(SOURCE_FILE)
)

df_empleados.printSchema()
display(df_empleados)

## Paso 3 — Crear tabla MANAGED en Unity Catalog

In [0]:
# Tabla MANAGED: UC controla la ubicacion fisica en ADLS
# DROP TABLE borrara los datos fisicos automaticamente
(
    df_empleados.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}.empleados_raw")
)

print(f"Tabla managed creada: {CATALOG}.{SCHEMA_BRONZE}.empleados_raw")

In [0]:
%sql
ALTER CATALOG dominio_data DISABLE PREDICTIVE OPTIMIZATION;

ALTER TABLE dominio_data.bronze.empleados_raw DISABLE PREDICTIVE OPTIMIZATION;



In [0]:
# Verificar que es MANAGED y ver la ubicacion gestionada por UC
# Buscar las filas: Type = MANAGED, Location = ruta gestionada
spark.sql("DESCRIBE EXTENDED dominio_data.bronze.empleados_raw").display()

## Paso 4 — Crear tabla EXTERNAL en Unity Catalog

In [0]:
%sql

DROP TABLE dominio_data.bronze.empleados_external

In [0]:
%sql

CREATE TABLE dominio_data.bronze.empleados_external
USING DELTA
LOCATION 'abfss://dominio-data@stacadbwus01.dfs.core.windows.net/external_tables/empleados_external'


In [0]:
# Tabla EXTERNAL: el ingeniero controla la ubicacion ADLS
# DROP TABLE solo borra el registro en el metastore; los archivos Delta permanecen
external_path = f"{TABLE_PATH}/empleados_external"

(
    df_empleados.write
    .format("delta")
    .mode("overwrite")
    .option("path", external_path)
    .saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}.empleados_external")
)

print(f"Tabla external creada apuntando a: {external_path}")

In [0]:
# Verificar que es EXTERNAL y ver que Location apunta al path ADLS explicitamente
spark.sql("DESCRIBE EXTENDED dominio_data.bronze.empleados_external").display()

### Diferencia clave para el examen

| Tipo | DROP TABLE | Datos en ADLS |
|------|-----------|---------------|
| **MANAGED** | Borra datos fisicos automaticamente | Se eliminan |
| **EXTERNAL** | Solo elimina el registro en el metastore | Permanecen |

Usar **MANAGED** para tablas transitorias o de proceso.  
Usar **EXTERNAL** para datos de produccion que deben sobrevivir al schema.

## Paso 5 — GRANT y REVOKE de permisos

In [0]:
# Ver el usuario actual
display(spark.sql("SELECT current_user() AS usuario_actual"))

In [0]:
# PATRON DE TRES CAPAS: para que un usuario lea una tabla en UC necesita:
# 1. USE CATALOG en el catalog
# 2. USE SCHEMA en el schema
# 3. SELECT en la tabla
#
# Reemplaza 'data.analyst@empresa.com' con un usuario real de tu workspace
# o con el nombre de un grupo existente.

# Paso 5A: USE CATALOG — prerequisito para cualquier acceso dentro del catalog
spark.sql("""
    GRANT USE CATALOG ON CATALOG dominio_data TO `docjmedina@dmclatam.onmicrosoft.com`
""").display()

# Paso 5B: USE SCHEMA — prerequisito para acceder a objetos del schema
spark.sql("""
    GRANT USE SCHEMA ON SCHEMA dominio_data.bronze TO `docjmedina@dmclatam.onmicrosoft.com`
""").display()

# Paso 5C: SELECT — permite leer la tabla
spark.sql("""
    GRANT SELECT ON TABLE dominio_data.bronze.empleados_raw TO `docjmedina@dmclatam.onmicrosoft.com`
""").display()

print("Descomentar los bloques GRANT para ejecutar con un usuario/grupo real.")
print("Los tres GRANT son necesarios — solo SELECT no es suficiente.")

In [0]:
# SHOW GRANTS: ver todos los permisos otorgados sobre la tabla
spark.sql("SHOW GRANTS ON TABLE dominio_data.bronze.empleados_raw").display()

In [0]:
spark.sql("""
    GRANT MODIFY ON TABLE dominio_data.bronze.empleados_raw TO `docjmedina@dmclatam.onmicrosoft.com`
""").display()


In [0]:
# REVOKE: revocar un permiso especifico sin afectar otros del mismo principal
spark.sql("""
    REVOKE MODIFY ON TABLE dominio_data.bronze.empleados_raw FROM `docjmedina@dmclatam.onmicrosoft.com`
""").display()

print("REVOKE quita un privilegio puntual sin afectar otros privilegios del mismo usuario.")

## Paso 6 — Auditoria con information_schema

In [0]:
# 6A: Listar columnas de la tabla con tipo de dato y posicion
spark.sql("""
    SELECT
        table_name,
        column_name,
        data_type,
        is_nullable,
        ordinal_position
    FROM dbassociate.information_schema.columns
    WHERE table_catalog = 'dbassociate'
      AND table_schema  = 'bronze'
    ORDER BY ordinal_position
""").display()

In [0]:
# 6B: Listar todas las tablas del catalog con su tipo (MANAGED / EXTERNAL)
spark.sql("""
    SELECT
        table_catalog,
        table_schema,
        table_name,
        table_type,
        created_by,
        last_altered
    FROM dbassociate.information_schema.tables
    WHERE table_catalog = 'dbassociate'
    ORDER BY table_schema, table_name
""").display()

In [0]:
# 6C: Ver privilegios otorgados en tablas del catalog
spark.sql("""
    SELECT
        grantor,
        grantee,
        table_catalog,
        table_schema,
        table_name,
        privilege_type,
        is_grantable
    FROM dbassociate.information_schema.table_privileges
    WHERE table_catalog = 'dbassociate'
    ORDER BY table_schema, table_name, grantee
""").display()

## Paso 7 — Limpieza

In [0]:
spark.sql("DROP TABLE IF EXISTS dbassociate.bronze.empleados_raw")
spark.sql("DROP TABLE IF EXISTS dbassociate.bronze.empleados_external")
dbutils.fs.rm(f"{TABLE_PATH}/empleados_external", recurse=True)
print("Limpieza completada.")

## Puntos clave del examen

1. La jerarquia UC es: **Metastore > Catalog > Schema > Tabla/Vista/Volumen/Funcion**
2. Para leer una tabla se necesitan **tres GRANTs**: USE CATALOG + USE SCHEMA + SELECT
3. `DROP TABLE` en una tabla **MANAGED** borra los datos fisicos; en una **EXTERNAL** solo borra el metadato
4. `information_schema` esta en cada catalog y expone metadata como tablas SQL
5. `SHOW GRANTS ON TABLE` lista todos los permisos vigentes sobre un objeto